# Market Differentiated Bars

## Process the Data

- **Purpose:** Apply fixed-width fractional differencing to dollar-bar log close prices.
- **Settings:** `weight_cutoff=0.01`; `d=0.0,0.1,…,1.0`; ADF 5% critical value; `maxlag=1`, `regression=c`, `autolag=None`.
- **Data:** Dollar-bar log close prices produce one persisted fractionally differenced feature.
- **Findings:**
- **Decision:** Select the smallest tested `d` that passes the ADF critical value using January 2025 preparation data only.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from src.preprocessing.market_data import ResearchPaths, load_manifest, read_raw

PROJECT_ROOT = Path.cwd().resolve().parents[1]
PERIOD = "2025-02-01_2025-12-31"
MANIFEST_PATH = PROJECT_ROOT / "data/preprocessing/universe/sp500_2025.csv"
EXPECTED_SECURITIES = 503
RESEARCH_START = pd.Timestamp("2025-02-01", tz="UTC")
WEIGHT_CUTOFF = 0.01
DIFFERENCING_ORDERS = np.linspace(0, 1, 11).tolist()
ADF_MAXLAG = 1
ADF_REGRESSION = "c"
ADF_AUTOLAG = None
ADF_SIGNIFICANCE = "5%"
paths = ResearchPaths(PROJECT_ROOT, period=PERIOD)
DISPLAY_SYMBOL = "AAPL"  # Inspection only; every manifest symbol is processed.

from src.preprocessing.market_differentiated_bars import build_fractional_features
display(build_fractional_features(
    paths,
    manifest_path=MANIFEST_PATH,
    expected_securities=EXPECTED_SECURITIES,
    fit_end=RESEARCH_START,
    weight_cutoff=WEIGHT_CUTOFF,
    differencing_orders=DIFFERENCING_ORDERS,
    adf_maxlag=ADF_MAXLAG,
    adf_regression=ADF_REGRESSION,
    adf_autolag=ADF_AUTOLAG,
    adf_significance=ADF_SIGNIFICANCE,
))
fractional_path = paths.feature(DISPLAY_SYMBOL, "fractional")
fractional_bars = pd.read_parquet(fractional_path)
fractional_path

## Take a Quick Look at the Data Structure

- **Purpose:** Inspect timestamps, dtype, range, and distribution of the persisted fractionally differenced feature.
- **Settings:**
- **Data:** Inspect the existing fractionally differenced feature artifact.
- **Findings:**
- **Decision:** Do not recalculate the feature or replace the selected differencing order.

In [ ]:
fractional_bars.head()

In [ ]:
fractional_bars.info()

In [ ]:
fractional_bars.dtypes.value_counts()

In [ ]:
fractional_bars[["fractionally_differenced_log_close"]].describe()

In [ ]:
fractional_bars[["fractionally_differenced_log_close"]].hist(
    bins=50, figsize=(6, 4)
)
plt.tight_layout()
plt.show()